In [1]:
import sys
sys.path.insert(0,'/nfs/nfs2/users/riadoshi/bigvision_palivla')
sys.path.insert(0,'/nfs/nfs2/users/riadoshi/bigvision_palivla/dlimp')
import json

import matplotlib.pyplot as plt

from octo.data.oxe import make_oxe_dataset_kwargs
from octo.data.dataset import make_single_dataset


2025-04-06 23:00:00.936171: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-06 23:00:00.941222: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-06 23:00:00.954037: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743980400.974512 3463620 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743980400.980693 3463620 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1743980400.996709 3463620 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [2]:
# load in aloha dataset
dataset = 'aloha_pick_place_full_dataset'
dataset_kwargs = make_oxe_dataset_kwargs(dataset,"gs://rail-orca-central2/resize_256_256/")
dataset_kwargs["use_cot"]=False
dataset = make_single_dataset(
                                dataset_kwargs, 
                                frame_transform_kwargs=dict(
                                    resize_size={"primary": (256, 256)},
                                ),
                                train=True)
iterator = dataset.iterator()

2025-04-06 23:00:07.723433: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


AttributeError: module 'ml_dtypes' has no attribute 'float4_e2m1fn'
Cause: Unable to locate the source code of <function _gcd_import at 0x7f8106c9bd80>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: Unable to locate the source code of <function _gcd_import at 0x7f8106c9bd80>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: Unable to locate the source code of <function _gcd_import at 0x7f8106c9bd80>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


2025-04-06 23:00:11.400679: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:387] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608


In [3]:
# load in pre-posthoc reasonings
with open('/nfs/nfs2/users/riadoshi/bigvision_palivla/data_generation/visualization/aloha/batch0.json', 'r') as f:
    reasoning_dct = json.load(f)


In [4]:
trajs = [next(iterator) for _ in range(10)]

In [7]:
for traj_id in range(5,10):
        traj = trajs[traj_id]
        reasoning = reasoning_dct[f'{traj_id}']
        images = traj['observation']['image_primary'].squeeze()

        import mediapy as media
        import numpy as np

        # Create a list to store all frames
        frames = []

        for step, image in enumerate(images):
        # Get right end effector coordinates

                if step % 10 == 0:
                        ry, rx = reasoning['end_effector_centroids']['right_end_effector'][f'{step}'].split(",")
                        
                        if ry != "None" and rx != "None":
                                ry, rx = float(ry), float(rx)
                        else:
                                ry, rx = 0, 0
                        
                        # Get left end effector coordinates
                        ly, lx = reasoning['end_effector_centroids']['left_end_effector'][f'{step}'].split(",")
                        
                        if ly != "None" and lx != "None":
                                ly, lx = float(ly), float(lx)
                        else:
                                ly, lx = 0, 0
                        
                        # Create a figure for this frame
                        fig, ax = plt.subplots(figsize=(4, 4))
                        ax.imshow(image)
                        
                        # Plot the end effector position
                        ax.scatter(rx, ry, c='red', s=100, marker='x')
                        ax.scatter(lx, ly, c='blue', s=100, marker='x')
                        ax.text(rx+10, ry+10, f'Right End Effector ({rx:.1f}, {ry:.1f})', color='white', fontsize=12, 
                                bbox=dict(facecolor='red', alpha=0.5))
                        ax.text(lx+10, ly+10, f'Left End Effector ({lx:.1f}, {ly:.1f})', color='white', fontsize=12, 
                                bbox=dict(facecolor='blue', alpha=0.5))
                        
                        # Add title
                        ax.set_title(f'Image {step} with End Effector Position')
                        ax.axis('on')
                        
                        # Convert figure to image array
                        fig.canvas.draw()
                        frame = np.array(fig.canvas.renderer.buffer_rgba())
                        frames.append(frame)
                        plt.close(fig)

        media.show_video(np.array(frames)[:, :, :, :3], fps=2)


This browser does not support the video tag.


This browser does not support the video tag.


This browser does not support the video tag.


This browser does not support the video tag.


KeyError: '9'